##  Enunciado do Case 1 - Modelo 1

Um trocador de calor de **correntes cruzadas** é utilizado para refrigerar um componente eletrônico de um equipamento de alta tecnologia, onde o controle de temperatura deve ter um rigor elevado.

O **fluido de trabalho** deste equipamento escoa a **M** litros por minuto [lpm] e deve ser resfriado de **X °C** para **Y °C** a fim de controlar o processamento do equipamento.

O **fluido refrigerante** (água) para realizar tal processo é alimentada no ponto triplo a **0 °C** e deixa o trocador de calor a **W °C**.

O coeficiente global de troca térmica do trocador é de **U W/m²·K**.

---

### Dados do fluido de trabalho

A densidade do fluido de trabalho é dada pela correlação empírica:

$$
\rho \left[ \frac{kg}{m^3} \right] = 0,0427 \cdot T^2 - 1,2337 \cdot T + 1058,6
$$

> **Válida para a faixa de temperatura de operação do equipamento.**

O calor específico do fluido de trabalho é constante na faixa de operação:

$$
c_p \left[ \frac{kJ}{kg \cdot K} \right] = 3,74
$$

---

### Dados para a água

As propriedades da água devem ser consultadas em **tabelas de líquido/vapor saturado** (Tabela A.6 – Incropera, 5ª ed.).

---

### Variáveis de entrada (função da matrícula)

Considere os **dois últimos dígitos** da matrícula para calcular as variáveis:

| Variável | Equação |
|----------|---------|
| **M** | $ M = 0,0435 \cdot matrícula + 5,9565 $ |
| **X** (Tce) | $ X = -0,0435 \cdot matrícula + 40,043 $ |
| **Y** (Tcs) | $ Y = 0,9273 \cdot \ln(matrícula) + 15,89 $ |
| **W** (Tts) | $ W = 0,058 \cdot matrícula + 12,942 $ |
| **U** | $ U = 0,0123 \cdot (matrícula)^2 - 0,1487 \cdot matrícula + 730,14 $ |

> **Onde:** $ n $ = dois últimos dígitos da matrícula.

---

### Objetivos do Case

Determinar:

- **(A)** A taxa de transferência de calor entre os fluidos;
- **(B)** A vazão mássica de água necessária;
- **(C)** A área de superfície do trocador de calor.


In [21]:
import math
import plotly.graph_objects as go

# DADOS DE ENTRADA
matricula = 17

M = 0.0435 * matricula + 5.9565          # L/min
X = -0.0435 * matricula + 40.043         # °C (Tce)
Y = 0.9273 * math.log(matricula) + 15.89 # °C (Tcs)
W = 0.058 * matricula + 12.942           # °C (Tts)
U = 0.0123 * matricula**2 - 0.1487 * matricula + 730.14  # W/(m²·K)

Tce = X
Tcs = Y
Tte = 0.0
Tts = W

cp_trab = 3.74   # kJ/(kg·K)

print("=== Dados calculados a partir da matrícula ===")
print(f"M = {M:.4f} L/min")
print(f"X (Tce) = {Tce:.4f} °C")
print(f"Y (Tcs) = {Tcs:.4f} °C")
print(f"W (Tts) = {Tts:.4f} °C")
print(f"U = {U:.4f} W/(m²·K)")

#  QUESTÃO A: TAXA DE CALOR 
Tm_trab = (Tce + Tcs) / 2
rho = 0.0427 * Tm_trab**2 - 1.2337 * Tm_trab + 1058.6  # kg/m³

V_dot = (M / 1000) / 60  # m³/s
m_trab = rho * V_dot      # kg/s

Q_kW = m_trab * cp_trab * (Tce - Tcs)  # kW
Q_W = Q_kW * 1000                      # W

print("\n=== A) Taxa de transferência de calor ===")
print(f"Tm,trab = {Tm_trab:.4f} °C")
print(f"rho = {rho:.4f} kg/m³")
print(f"V_dot = {V_dot:.6e} m³/s")
print(f"m_trab = {m_trab:.6f} kg/s")
print(f"Q = {Q_kW:.4f} kW = {Q_W:.2f} W")

=== Dados calculados a partir da matrícula ===
M = 6.6960 L/min
X (Tce) = 39.3035 °C
Y (Tcs) = 18.5172 °C
W (Tts) = 13.9280 °C
U = 731.1668 W/(m²·K)

=== A) Taxa de transferência de calor ===
Tm,trab = 28.9104 °C
rho = 1058.6223 kg/m³
V_dot = 1.116000e-04 m³/s
m_trab = 0.118142 kg/s
Q = 9.1845 kW = 9184.45 W


In [ ]:
# INTERPOLAÇÃO DO Cp DA ÁGUA 
Tm_agua = (Tte + Tts) / 2          # °C
Tm_agua_K = Tm_agua + 273.15       # K (280.114 K)

# Dados DA TABELA A.6 (Incropera, 5ª ed.) ENTR 275 K e 280 K
T1, cp1 = 275.0, 4.211   # kJ/(kg·K)
T2, cp2 = 280.0, 4.198   # kJ/(kg·K)

cp_agua = cp1 + (cp2 - cp1) / (T2 - T1) * (Tm_agua_K - T1)

print(f"\nTm,água = {Tm_agua:.3f} °C = {Tm_agua_K:.3f} K")
print(f"cp_água (interpolado) = {cp_agua:.4f} kJ/(kg·K)")

#  QUESTÃO B: VAZÃO DE ÁGUA 
m_agua = Q_kW / (cp_agua * (Tts - Tte))  # kg/s

print("\n=== B) Vazão mássica de água ===")
print(f"m_agua = {m_agua:.4f} kg/s")

# QUESTÃO C: ÁREA 
# DTML
dTa = Tce - Tts
dTb = Tcs - Tte
DTML = (dTa - dTb) / math.log(dTa / dTb)

# PÂRAMETROS Z e P
Z = (Tce - Tcs) / (Tts - Tte)
P = (Tts - Tte) / (Tce - Tte)

# FATOR DE CORREÇÃO (WebPlotDigitizer)
F = 0.9325

# Área
A = Q_W / (U * F * DTML)  # m²

print("\n=== C) Área da superfície do trocador ===")
print(f"DTML = {DTML:.4f} °C")
print(f"Z = {Z:.4f}")
print(f"P = {P:.4f}")
print(f"F (WebPlotDigitizer) = {F:.4f}")
print(f"A = {A:.4f} m²")

print("\n" + "="*50)
print("RESUMO FINAL (S.I.)")
print("="*50)
print(f"Q = {Q_W:.1f} W")
print(f"m_água = {m_agua:.4f} kg/s")
print(f"A = {A:.4f} m²")
print("="*50)


Tm,água = 6.964 °C = 280.114 K
cp_água (interpolado) = 4.1977 kJ/(kg·K)

=== B) Vazão mássica de água ===
m_agua = 0.1571 kg/s

=== C) Área da superfície do trocador ===
DTML = 21.7666 °C
Z = 1.4924
P = 0.3544
F (WebPlotDigitizer) = 0.9325
A = 0.6189 m²

RESUMO FINAL (S.I.)
Q = 9184.5 W
m_água = 0.1571 kg/s
A = 0.6189 m²


In [ ]:
# GRÁFICO 
Tce = 39.3035
Tcs = 18.5172
Tte = 0.0
Tts = 13.928

dTa = Tce - Tts
dTb = Tcs - Tte

x = [0, 1]
y_carcaca = [Tce, Tcs]
y_tubos = [Tts, Tte]


# APENAS PLOTAGEM DE GRÁFICO (SEM CÁLCULOS)
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x, y=y_carcaca,
    mode='lines+markers',
    name='Fluido de trabalho (casco)',
    line=dict(color='firebrick', width=3),
    marker=dict(size=10, color='firebrick')
))

fig.add_trace(go.Scatter(
    x=x, y=y_tubos,
    mode='lines+markers',
    name='Água (tubos)',
    line=dict(color='royalblue', width=3),
    marker=dict(size=10, color='royalblue')
))

fig.add_annotation(
    x=0.5, y=(Tce + Tts)/2,
    text=f'ΔTₐ = {dTa:.3f} °C',
    showarrow=True, arrowhead=1, ax=40, ay=0,
    font=dict(color='gray', size=12)
)

fig.add_annotation(
    x=0.5, y=(Tcs + Tte)/2,
    text=f'ΔT_b = {dTb:.3f} °C',
    showarrow=True, arrowhead=1, ax=-40, ay=0,
    font=dict(color='gray', size=12)
)

fig.update_layout(
    title=dict(
        text='Perfil de temperaturas equivalente em contracorrente<br><sup>Usado para o cálculo de ΔTml</sup>',
        font=dict(size=16, family='Arial Black'),
        x=0.5
    ),
    xaxis=dict(
        title='Posição relativa ao longo da área de troca',
        tickvals=[0, 1],
        ticktext=['Entrada carcaça / Saída tubos', 'Saída carcaça / Entrada tubos']
    ),
    yaxis_title='Temperatura (°C)',
    legend=dict(x=0.02, y=0.98),
    width=800, height=500,
    plot_bgcolor='white'
)

fig.show()